# 05 — PAD Dataset EDA

Quick look at the NUAA Imposter Database before training. Goal: confirm class balance, spot any image-quality issues, and verify the dataset class is reading the folders correctly.

Run this from the repo root after `data/pad/` is populated.

In [ ]:
# imports + path setup
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

from src.config import PAD_DIR
from src.data.pad_dataset import PADDataset

sns.set_theme(style='whitegrid')
print(f'PAD folder: {PAD_DIR}')
print(f'exists: {PAD_DIR.exists()}')

In [ ]:
# load dataset, no transforms — labels and paths only
ds = PADDataset(transform=None)
print(f'total samples: {len(ds):,}')
print(f'class counts: {ds.class_counts()}')

## 1. Class balance

Imbalance here drives the choice of loss weights and the metric we use. NUAA is roughly 50/50, but client and imposter folders may not match exactly.

In [ ]:
counts = ds.class_counts()
df = pd.DataFrame({'class': list(counts.keys()), 'count': list(counts.values())})

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.barplot(data=df, x='class', y='count', ax=ax, color='steelblue')
ax.set_title('PAD — class balance (NUAA Imposter DB)')
for i, v in enumerate(df['count']):
    ax.text(i, v, f'{v:,}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

ratio = counts['real'] / max(counts['attack'], 1)
print(f'real:attack ratio = {ratio:.2f}:1')

## 2. Sample images — visual sanity check

Spot any obvious labeling problems (real folder containing printed photos or vice versa).

In [ ]:
rng = np.random.default_rng(42)

# 4 real, 4 attack — equal coverage of each class
real_idxs = [i for i, s in enumerate(ds.samples) if s.label == 0]
attack_idxs = [i for i, s in enumerate(ds.samples) if s.label == 1]
pick_real = rng.choice(real_idxs, size=4, replace=False)
pick_attack = rng.choice(attack_idxs, size=4, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes[0], pick_real):
    item = ds[int(idx)]
    ax.imshow(Image.open(item['path']).convert('RGB'))
    ax.set_title('real', color='green')
    ax.axis('off')
for ax, idx in zip(axes[1], pick_attack):
    item = ds[int(idx)]
    ax.imshow(Image.open(item['path']).convert('RGB'))
    ax.set_title('attack', color='red')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Image dimensions

NUAA images vary in size. The transform pipeline resizes to 224x224 — good to know what is being downscaled vs upscaled going in.

In [ ]:
# sample 200 images, record their original dimensions
sample_idxs = rng.choice(len(ds), size=min(200, len(ds)), replace=False)
rows = []
for idx in sample_idxs:
    p = ds.samples[int(idx)].path
    with Image.open(p) as im:
        w, h = im.size
    rows.append({'path': str(p), 'width': w, 'height': h, 'label': ds.samples[int(idx)].label})
dim_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(dim_df['width'], bins=30, ax=axes[0])
axes[0].set_title('image width distribution')
sns.histplot(dim_df['height'], bins=30, ax=axes[1])
axes[1].set_title('image height distribution')
plt.tight_layout()
plt.show()

print(f'width  range: {dim_df["width"].min()} - {dim_df["width"].max()}')
print(f'height range: {dim_df["height"].min()} - {dim_df["height"].max()}')

## 4. Findings — what to remember for training

Fill in after running:

- class balance: ...
- visible label noise: ...
- image size range: ...

These notes go into the README's Tech Stack notes for PAD.